[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/en_clase/S05_01_numpy_en_clase.ipynb)

# S05_01 · NumPy: arrays y vectorización (en clase)

<h2 align="center">Trabajar con 1096 días de datos a la vez, sin escribir bucles</h2>

| | |
|---|---|
| ⏱ **Cuándo** | Miércoles · unos 30–35 min |
| 👥 **Cómo** | En parejas: una persona escribe, la otra lee y comprueba (cambiad de papel al empezar la parte B) |
| 🧰 **Usáis** | 🌐 Google Colab (este cuaderno) · no hay que instalar nada |
| 🏁 **Al terminar** | Ejercicio 1 (arrays) y ejercicio 3 (vectorización) con todo ✅ en el paso 5 |

**Este cuaderno tiene dos partes:**

- **1 · Guía paso a paso**: solo lo que hay que hacer. Va en dos bloques: **A · Arrays** y **B · Vectorización**.
- **2 · Explicaciones**: el porqué de cada cosa, para cuando lo necesitéis (está al final, también en dos bloques: A y B).
- Si queréis, seguid solo la parte 1.

**Solo trae los dos ejercicios de clase**, cada uno en su propia parte: el **1 trata de arrays** (parte A) y el **3 de vectorización** (parte B). Los demás (2, 4, 5 y el mini-reto) están en el cuaderno completo [`S05_01_numpy`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/para_casa/S05_01_numpy.ipynb), para casa.

## Cómo se usa este cuaderno (leedlo una vez)

**Un cuaderno es una mezcla de textos y celdas de código.** Las celdas de código tienen a la izquierda un botón **▶**.

1. **Ejecutar una celda**
   - Pulsad el botón **▶** de la izquierda, o haced clic dentro de la celda y pulsad **Mayús + Intro**.
   - Debajo de la celda aparece **lo que ha salido**.
2. **De arriba abajo, en orden**
   - Cada celda usa lo que han hecho las anteriores.
   - Si os saltáis una, saldrá un error rojo que dice `NameError`. No pasa nada: ejecutad las que faltan y repetid.
3. **Las celdas con `...`**
   - Son los ejercicios. Los **tres puntos `...` se sustituyen** por vuestro código.
   - No borréis el resto de la celda.
4. **Cada paso termina en ✅**
   - Es lo que **tiene que salir**.
   - Los decimales lejanos o los milisegundos pueden cambiar un poco en vuestro ordenador; lo demás tiene que ser **igual**.
   - Si no sale, no sigáis: mirad la parte 2, apartado «Si os atascáis».
5. **Si Colab se reinicia o cerráis la pestaña**
   - Se pierde lo ejecutado, no lo escrito.
   - Volved a ejecutar las celdas **desde el paso 0 hasta donde estabais**.

# 1 · Guía paso a paso

**Qué vais a hacer**

- **Parte A · Arrays (ejercicio 1):** cargar una tabla con **1096 días** de un alojamiento de Tenerife (ocupación, temperatura, festivo y precio) y separarla en sus cuatro columnas.
- **Parte B · Vectorización (ejercicio 3):** calcular el ingreso de cada día y «estandarizar» la ocupación, **todo de una vez, sin bucles**.

## Antes de empezar · Paso 0 · Preparar el cuaderno (3 min)

| # | Qué hacéis |
|---|---|
| **0.1** | Ejecutad la celda de abajo (la de «Preparación»). Tarda unos segundos: busca el fichero de datos y, si no está, lo descarga. |
| **0.2** | Ejecutad la celda siguiente (la de las librerías). |

✅ Con la primera sale `Preparación lista ✅` y con la segunda `NumPy listo, versión 2.…` (el número puede cambiar).

*No hace falta entender estas dos celdas: dejan preparado lo que usaremos después.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

RESULTADOS = {}

def comprobar(nombre, condicion, ayuda):
    """Escribe ✅ si la condición es cierta y ❌ con una pista si no. No hace falta entender cómo funciona."""
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    RESULTADOS[nombre] = ok
    print(("✅ " + nombre) if ok else ("❌ " + nombre + " → " + ayuda))

dato("ocupacion_diaria.csv")
print("Preparación lista ✅")

In [ ]:
import time

import numpy as np

print("NumPy listo, versión", np.__version__)

# Parte A · Arrays (ejercicio 1) · unos 15 min

**Qué es un array y qué vais a hacer con él**

- Un **array** es una **tabla de números**: así guarda NumPy los datos.
- Vais a **cargar** la tabla de 1096 días y a aprender a **pedir un trozo**: una columna, una fila o un solo valor.
- Después, el **ejercicio 1**: dar a cada columna su propio nombre.
- La parte A **no hace cuentas**: solo carga y selecciona datos. Las cuentas son de la parte B.

## Paso 1 · Cargar la tabla y mirarla (5 min)

**Los datos**

- Una **fila por día**, del 1/1/2023 al 31/12/2025.
- **Cuatro columnas**, que se cuentan desde el **0**:

| Columna | Qué es | Ejemplo (1.er día) |
|---|---|---|
| **0** | Ocupación (%) | 89.0 |
| **1** | Temperatura media (°C) | 18.1 |
| **2** | Festivo (1 = sí, 0 = no) | 1 |
| **3** | Precio medio de la habitación (€) | 97.43 |

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **1.1** | Ejecutad la celda 1.1 (carga el fichero en una variable llamada `tabla`). | `forma: (1096, 4) · tipo: float64 · dimensiones: 2` |
| **1.2** | Ejecutad la celda 1.2 (muestra las 3 primeras filas). | Tres filas de cuatro números; la primera empieza por `89.` |
| **1.3** | Ejecutad la celda 1.3 (muestra **una columna entera**, la 0). | `[89.  84.7 71.6 85.6 78.7]` |
| **1.4** | Ejecutad la celda 1.4 (muestra **un solo valor**: fila 0, columna 1). | `18.1` |

- `(1096, 4)` quiere decir: **1096 filas** (días) y **4 columnas**.
- Mirad cómo se escribe: `tabla[filas, columnas]`. Los dos puntos `:` significan **«todas»**.

In [ ]:
# 1.1 · Cargar la tabla
tabla = np.genfromtxt(dato("ocupacion_diaria.csv"), delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
print("forma:", tabla.shape, "· tipo:", tabla.dtype, "· dimensiones:", tabla.ndim)

In [ ]:
# 1.2 · Las tres primeras filas
print(tabla[:3])

In [ ]:
# 1.3 · Una columna entera (la 0): tabla[todas las filas, columna 0]
print(tabla[:5, 0])      # solo enseñamos las 5 primeras para que quepa

In [ ]:
# 1.4 · Un solo valor: fila 0, columna 1
print(tabla[0, 1])

## Paso 2 · Ejercicio 1 · Separar las columnas (8 min) 🏫

**Objetivo**

- Que cada columna tenga su propio nombre.
- Así no hay que recordar qué número es cada cosa.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **2.1** | Mirad la tabla del paso 1: la columna **0** es la ocupación, la **1** la temperatura, la **2** el festivo y la **3** el precio. | — |
| **2.2** | Ejecutad la celda 2.2: es **un ejemplo ya hecho**, con la ocupación. | `[89.  84.7 71.6 85.6 78.7]` |
| **2.3** | **Vuestro turno.** En la celda 2.3, sustituid cada `...` por `tabla[:, k]`, cambiando **k** por el número de la columna que toca. Es igual que el ejemplo, cambiando solo el número. | No sale nada (solo se guardan las variables) |
| **2.4** | Ejecutad la celda 2.4 (la comprobación). | Cuatro líneas con ✅ |

- **Una pista para 2.3:** `temperatura` es la columna **1**, `festivo` la **2** y `precio` la **3**.
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.

In [ ]:
# 2.2 · Ejemplo ya hecho: la ocupación es la columna 0
ocupacion = tabla[:, 0]
print(ocupacion[:5])

In [ ]:
# 2.3 · TU TURNO: sustituye cada ... por tabla[:, k] (k = número de la columna)
temperatura = ...
festivo = ...
precio = ...

In [ ]:
# 2.4 · Comprobación del ejercicio 1 (no hay que cambiar nada)
comprobar("ocupacion", lambda: ocupacion.shape == (1096,), "debe ser la columna 0 entera: tabla[:, 0]")
comprobar("temperatura", lambda: temperatura.mean().round(2) == 21.10, "debe ser la columna 1: tabla[:, 1]. ¿Has cambiado los puntos ... por tu código?")
comprobar("festivo", lambda: festivo.sum() == 61, "debe ser la columna 2: tabla[:, 2]. ¿Has cambiado los puntos ... por tu código?")
comprobar("precio", lambda: precio.mean().round(2) == 87.52, "debe ser la columna 3: tabla[:, 3]. ¿Has cambiado los puntos ... por tu código?")

✅ Tiene que salir: `✅ ocupacion`, `✅ temperatura`, `✅ festivo` y `✅ precio`. Si falta alguno, no sigáis.

# Parte B · Vectorización (ejercicio 3) · unos 15 min

*Cambiad de papel: quien escribía, ahora lee y comprueba.*

**Qué es vectorizar y qué vais a hacer**

- **Vectorizar** es hacer una cuenta **con todos los datos a la vez**, sin escribir un bucle que vaya uno por uno.
- Primero **veréis la diferencia** con un millón de números (paso 3).
- Después, el **ejercicio 3**: el ingreso de cada día y la ocupación estandarizada, con una sola línea cada una (paso 4).
- Para esta parte se usan las columnas que separasteis en la parte A: si no las tenéis, volved al paso 2.

## Paso 3 · Ver la diferencia: con bucle y sin bucle (3 min) 🏫

**Objetivo**

- Ver con vuestros ojos por qué NumPy se usa tanto.
- Se hace **el mismo cálculo** con un millón de números, de dos maneras.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **3.1** | Ejecutad la celda 3.1. Tarda un par de segundos. | `Bucle: … ms · vectorizado: … ms · el resultado es el mismo: True` |
| **3.2** | Comparad los dos tiempos. | El vectorizado es **decenas o cientos de veces más rápido** (los milisegundos exactos cambian en cada ordenador) |

- **Con bucle:** un `for` va número por número (un millón de vueltas).
- **Vectorizado:** `grande * 2` lo hace **todo de una vez**.
- El resultado es **el mismo**: solo cambia lo que se tarda y lo corto que se escribe.

In [ ]:
# 3.1 · Un millón de números, con bucle y sin bucle
grande = np.arange(1_000_000, dtype=float)

inicio = time.perf_counter()
suma_bucle = 0.0
for valor in grande.tolist():
    suma_bucle += valor * 2
t_bucle = time.perf_counter() - inicio

inicio = time.perf_counter()
suma_vector = (grande * 2).sum()
t_vector = time.perf_counter() - inicio

print(f"Bucle: {t_bucle*1000:.1f} ms · vectorizado: {t_vector*1000:.2f} ms · el resultado es el mismo: {suma_bucle == suma_vector}")

## Paso 4 · Ejercicio 3 · Ingreso diario y estandarización (12 min) 🏫

**Qué se pide**

- **Primero:** el **ingreso de cada día** de un alojamiento de **120 habitaciones**: `ocupacion / 100 × 120 × precio`.
- **Después:** **estandarizar** la ocupación (se explica abajo, con un ejemplo de 4 números).

### Primero · El ingreso de cada día

**Por qué sale esa fórmula**

- `ocupacion / 100`: pasa el porcentaje a un número entre 0 y 1 (89 % → 0.89).
- `× 120`: cuántas **habitaciones ocupadas** hay (0.89 × 120 = 106.8).
- `× precio`: cuánto **dinero** dejan (106.8 × 97.43 = 10405.52 €).

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.1** | **A mano, con un solo día.** Ejecutad la celda 4.1: hace la cuenta **solo del primer día**. | `10405.52` |
| **4.2** | **Vuestro turno.** En la celda 4.2, sustituid `...` por **la misma cuenta, pero sin `[0]`**: así se hace con **todos los días a la vez**. | No sale nada |
| **4.3** | Ejecutad la celda 4.3 (la comprobación). | Dos líneas con ✅ |
| **4.4** | Ejecutad la celda 4.4 (mira los 3 primeros ingresos y la media). | `[10405.52  9891.6   8345.41]` y `Ingreso medio diario: 7,237 EUR` |

- **Pista para 4.2:** `ingreso_diario = ocupacion / 100 * 120 * precio`. Con `ocupacion` y `precio` **sin corchetes**, NumPy hace la cuenta **día a día** sin bucle.

In [ ]:
# 4.1 · A mano, solo el primer día: ocupacion[0] y precio[0]
print(round(ocupacion[0] / 100 * 120 * precio[0], 2))

In [ ]:
# 4.2 · TU TURNO: la misma cuenta, pero con ocupacion y precio enteros (sin [0])
ingreso_diario = ...

In [ ]:
# 4.3 · Comprobación del ingreso diario (no hay que cambiar nada)
comprobar("ingreso_diario: un valor por día", lambda: ingreso_diario.shape == (1096,), "debe haber 1096 valores, uno por día: no pongas [0]. ¿Has cambiado los puntos ... por tu código?")
comprobar("ingreso_diario: primer día", lambda: np.isclose(ingreso_diario[0], 10405.524), "el primer día debe salir 10405.52: ocupacion / 100 * 120 * precio")

In [ ]:
# 4.4 · Mirar el resultado
print(ingreso_diario[:3].round(2))
print(f"Ingreso medio diario: {ingreso_diario.mean():,.0f} EUR")

### Después · Estandarizar la ocupación

**Qué es estandarizar**

- Es cambiar los números para que su **media sea 0** y su **dispersión sea 1**.
- Se hace en dos pasos: **restar la media** y **dividir entre la desviación típica** (la desviación típica mide cuánto se separan los datos de la media).
- El resultado dice **a cuántas desviaciones de la media** está cada dato. Se llama `z`.
- Sirve para comparar cosas con unidades distintas (%, °C, €) en la misma escala.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.5** | **A mano, con 4 números** (2, 4, 6 y 8). Ejecutad la celda 4.5. | `5.0 2.24` y `[-1.34 -0.45  0.45  1.34]` |
| **4.6** | **Vuestro turno.** En la celda 4.6, sustituid `...` por la misma cuenta aplicada a la ocupación: `(ocupacion - ocupacion.mean()) / ocupacion.std()`. | No sale nada |
| **4.7** | Ejecutad la celda 4.7 (la comprobación). | Dos líneas con ✅ |
| **4.8** | Ejecutad la celda 4.8 (mira las 3 primeras ocupaciones estandarizadas). | `[2.13 1.69 0.32]` |

- En 4.5: la media es 5. El **2** queda a −1.34 desviaciones de la media; el **8**, a +1.34.
- La ocupación del primer día (89 %) queda en **2.13**: muy por encima de la media.

In [ ]:
# 4.5 · A mano, con cuatro números: 2, 4, 6 y 8
numeros = np.array([2, 4, 6, 8])
media = numeros.mean()                 # el centro: 5.0
desviacion = numeros.std()             # cuánto se separan de la media: 2.24
print(media, desviacion.round(2))
print(((numeros - media) / desviacion).round(2))     # restar la media y dividir entre la desviación

In [ ]:
# 4.6 · TU TURNO: restar la media y dividir entre la desviación, con la ocupación entera
ocupacion_z = ...

In [ ]:
# 4.7 · Comprobación de la estandarización (no hay que cambiar nada)
comprobar("ocupacion_z: media 0", lambda: abs(ocupacion_z.mean()) < 1e-9, "la media debe salir 0: (ocupacion - ocupacion.mean()) / ocupacion.std(). ¿Has cambiado los puntos ... por tu código?")
comprobar("ocupacion_z: desviación 1", lambda: abs(ocupacion_z.std() - 1) < 1e-9, "la desviación debe salir 1: divide entre ocupacion.std()")

In [ ]:
# 4.8 · Mirar el resultado
print(ocupacion_z[:3].round(2))

## Paso 5 · Cerrar los dos ejercicios (2 min)

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **5.1** | Ejecutad la celda 5.1 (cuenta las comprobaciones ✅ de cada ejercicio por separado). | `Parte A · Arrays (ejercicio 1): 4 de 4 ✅`, `Parte B · Vectorización (ejercicio 3): 4 de 4 ✅` y `🎉 Los ejercicios 1 y 3 están hechos` |
| **5.2** | Si os sobra tiempo, abrid el cuaderno completo [`S05_01_numpy`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/para_casa/S05_01_numpy.ipynb) y probad los ejercicios 2, 4 y 5. | — |

In [ ]:
# 5.1 · Resumen por ejercicio
N1 = ["ocupacion", "temperatura", "festivo", "precio"]
N3 = ["ingreso_diario: un valor por día", "ingreso_diario: primer día", "ocupacion_z: media 0", "ocupacion_z: desviación 1"]
bien1 = sum(RESULTADOS.get(n, False) for n in N1)
bien3 = sum(RESULTADOS.get(n, False) for n in N3)
print(f"Parte A · Arrays (ejercicio 1): {bien1} de 4 ✅")
print(f"Parte B · Vectorización (ejercicio 3): {bien3} de 4 ✅")
if bien1 == 4 and bien3 == 4:
    print("🎉 Los ejercicios 1 y 3 están hechos")
else:
    print("Faltan comprobaciones: mirad las líneas ❌ de las celdas 2.4 (ejercicio 1) y 4.3 y 4.7 (ejercicio 3)")

## ✅ Terminado cuando

**Parte A · Arrays (ejercicio 1)**

- [ ] La celda 2.4 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué hace `tabla[:, 1]`**.

**Parte B · Vectorización (ejercicio 3)**

- [ ] Las celdas 4.3 y 4.7 muestran todo ✅.
- [ ] Cada persona sabe explicar con sus palabras **por qué no hace falta un bucle** para calcular el ingreso de 1096 días.

**Los dos**

- [ ] La celda 5.1 dice `🎉 Los ejercicios 1 y 3 están hechos`.

---

# 2 · Explicaciones

*Para cuando una cosa no sale o queréis entender por qué se hace así. No hace falta leerlo para resolver los ejercicios. Está en dos bloques, igual que la guía: **A · Arrays** y **B · Vectorización**.*

# Explicaciones de la parte A · Arrays (ejercicio 1)

## La idea: un array es una tabla de números

- Un **array** de NumPy es una **tabla de números del mismo tipo**, como una hoja de cálculo sin textos.
- NumPy es una **librería**: un conjunto de herramientas ya hechas para trabajar con números. Se importa con `import numpy as np` (así la llamamos `np`).
- Un array tiene tres datos básicos:
  - **`shape` (forma):** cuántas filas y columnas. Aquí `(1096, 4)`.
  - **`dtype` (tipo):** qué clase de números guarda. Aquí `float64` (decimales).
  - **`ndim` (dimensiones):** 1 si es una lista simple, 2 si es una tabla.

## Cómo se lee `tabla[filas, columnas]`

| Escribo | Significa | Resultado |
|---|---|---|
| `tabla[0, 1]` | Fila 0, columna 1 | `18.1` (un valor) |
| `tabla[:, 0]` | **Todas** las filas, columna 0 | La ocupación de los 1096 días |
| `tabla[:3, 0]` | Las **3 primeras** filas, columna 0 | `[89.  84.7 71.6]` |
| `tabla[10:13, :2]` | Filas 10, 11 y 12; las **2 primeras** columnas | Una tabla de 3 filas y 2 columnas |

- Los **dos puntos `:`** quieren decir «todo»; `a:b` quiere decir «desde `a` hasta `b`, sin incluir `b`».
- **Se cuenta desde 0:** la primera fila es la 0 y la primera columna es la 0. Por eso la columna del precio es la **3** y no la 4 (si escribís `tabla[:, 4]` sale `IndexError`).

## Las palabras de la parte A

| Palabra | Qué significa |
|---|---|
| **Array** | Una tabla de números del mismo tipo (la de NumPy) |
| **Librería (`np`)** | Un conjunto de herramientas ya hechas; `np` es el nombre corto de NumPy |
| **Variable** | Un nombre que guarda un dato: `ocupacion = tabla[:, 0]` guarda ahí la columna 0 |
| **`shape`** | Su forma: `(filas, columnas)` |
| **Índice** | El número de posición de un dato; **empieza en 0** |
| **Corte (`:`)** | Elegir un trozo: `3:7` son los puestos 3, 4, 5 y 6 |
| **Columna / fila** | Una línea vertical / horizontal de la tabla |
| **`print`** | Enseña un valor por pantalla |
| **✅ / ❌** | Lo que escribe la función `comprobar`: ✅ está bien; ❌ hay algo que corregir (con una pista) |

## Por qué se hace así

- **¿Por qué dar nombre a cada columna (ejercicio 1)?**
  - `ocupacion` se lee mejor que `tabla[:, 0]`.
  - Si no, habría que recordar qué número es cada cosa, y es fácil equivocarse.
- **¿Por qué se cuenta desde 0?**
  - Es la costumbre de casi todos los lenguajes de programación: el índice dice **cuántos puestos hay antes**. Delante de la primera no hay ninguno.
- **¿Por qué `tabla[:, 1]` y no `tabla[1]`?**
  - `tabla[1]` es la **fila** 1 (el segundo día, con sus cuatro datos).
  - `tabla[:, 1]` es la **columna** 1 (la temperatura de todos los días).

# Explicaciones de la parte B · Vectorización (ejercicio 3)

## La idea: vectorizar es hacer la cuenta con todos a la vez

- Imaginad que hay que **sumar 2** a un millón de números.
  - **Con bucle:** cogéis un número, le sumáis 2, lo apuntáis; cogéis el siguiente… un millón de veces.
  - **Vectorizado:** escribís `numeros + 2` y NumPy lo hace **con todos a la vez**.
- Se llama «vectorizar» porque se trabaja con el array entero (un «vector») y no con cada número suelto.
- El resultado es **el mismo**; cambia lo corto que se escribe y lo mucho que se tarda.

## La fórmula del ingreso, paso a paso

- **`ocupacion / 100`:** pasa el porcentaje a un número entre 0 y 1 (89 % → 0.89).
- **`× 120`:** cuántas habitaciones ocupadas hay (0.89 × 120 = 106.8). Las 120 habitaciones son un **dato supuesto** del ejercicio: no salen del fichero.
- **`× precio`:** cuánto dinero dejan (106.8 × 97.43 = 10405.52 €).
- Con `ocupacion` y `precio` **enteros** (sin `[0]`), NumPy repite la cuenta **día a día** y devuelve un array con **1096 ingresos**.

## Estandarizar, con el ejemplo de 4 números

- Los datos son **2, 4, 6 y 8**.
- Su **media** es 5.0.
- Su **desviación típica** es 2.24.

| Dato | Menos la media | Entre la desviación | Es `z` |
|---|---|---|---|
| 2 | 2 − 5 = −3 | −3 ÷ 2.24 | **−1.34** |
| 4 | 4 − 5 = −1 | −1 ÷ 2.24 | **−0.45** |
| 6 | 6 − 5 = 1 | 1 ÷ 2.24 | **0.45** |
| 8 | 8 − 5 = 3 | 3 ÷ 2.24 | **1.34** |

- **Restar la media** centra los datos: ahora el centro es 0.
- **Dividir entre la desviación** pone todos en la misma escala: ahora la dispersión es 1.
- `z` dice **a cuántas desviaciones de la media** está cada dato.
- En NumPy se hace con los cuatro a la vez: `(numeros - media) / desviacion`.

## Las palabras de la parte B

| Palabra | Qué significa |
|---|---|
| **Bucle (`for`)** | Repetir algo número por número |
| **Vectorizar** | Hacer la operación **sobre todo el array a la vez**, sin bucle |
| **Media (`.mean()`)** | La suma de los datos dividida entre cuántos son |
| **Desviación típica (`.std()`)** | Cuánto se separan los datos de la media, de media |
| **Estandarizar** | Restar la media y dividir entre la desviación típica: queda media 0 y desviación 1 |
| **`z`** | Lo que sale al estandarizar: a cuántas desviaciones de la media está un dato |
| **`.round(2)`** | Redondea a 2 decimales |
| **Milisegundo (ms)** | Una milésima de segundo: el paso 3 mide el tiempo con esa unidad |

## Por qué se hace así

- **¿Por qué no un bucle (paso 3 y ejercicio 3)?**
  - Más corto: `ocupacion / 100 * 120 * precio` en lugar de un `for` con varias líneas.
  - Más rápido: NumPy trabaja con todos los números a la vez, en código muy optimizado. Con un millón de números, el bucle tarda cientos de veces más.
  - Se parece más a una fórmula de matemáticas: es lo que luego se usa en IA.
- **¿Por qué `ocupacion / 100` y no solo `ocupacion`?**
  - La ocupación viene en porcentaje (89). Para multiplicar por las habitaciones hace falta una fracción (0.89).
  - Si se olvida, el ingreso sale **100 veces mayor**.
- **¿Por qué estandarizar?**
  - Las columnas tienen unidades distintas: % de ocupación, °C, €.
  - Con `z` todas se miden en «desviaciones de la media» y se pueden comparar.
  - Los modelos de IA que veréis más adelante lo piden a menudo.
- **¿Por qué la media de `ocupacion_z` sale «casi» 0 y no exactamente 0?**
  - Los ordenadores guardan los decimales con un error minúsculo (del orden de 0,000000000000001).
  - Por eso la comprobación acepta valores **muy cercanos** a 0 y no exactamente 0.

## 🚑 Si os atascáis

**Parte A · Arrays (ejercicio 1)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `❌ temperatura → …` (o `festivo`, `precio`) | La variable no tiene la columna que debe | Mirad la pista de la flecha `→`: `temperatura` es la columna 1, `festivo` la 2 y `precio` la 3 |
| `IndexError: index 4 is out of bounds` o `too many indices` | Se ha pedido una columna que no existe: solo hay 0, 1, 2 y 3 | Usad una columna de 0 a 3 |
| `forma: (1096,)` o una tabla de 4 filas en vez de 1096 valores | Se ha escrito `tabla[k]` en vez de `tabla[:, k]` | Poned el `:,` delante: `tabla[:, 1]` |
| `AttributeError` o `TypeError` que nombran `'ellipsis'` | Se ha quedado algún `...` sin sustituir | Cambiad los `...` por vuestro código |

**Parte B · Vectorización (ejercicio 3)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'ocupacion' is not defined` | No se ha hecho el ejercicio 1: la parte B usa sus columnas | Volved al paso 2 y ejecutad la celda 2.2 y la 2.3 |
| El ingreso medio sale unos `723 658` en lugar de `7 237` | Falta el `/ 100` de la ocupación | Escribid `ocupacion / 100 * 120 * precio` |
| `ingreso_diario` es un solo número, no 1096 | Se ha dejado `[0]` en la cuenta (`ocupacion[0]`) | Quitad `[0]`: `ocupacion` y `precio` enteros |
| `❌ ocupacion_z: desviación 1` | Se ha restado la media pero no se ha dividido | `(ocupacion - ocupacion.mean()) / ocupacion.std()`, con los **paréntesis** |
| `TypeError: 'ellipsis' object is not subscriptable` en 4.4 o 4.8 | La celda del ejercicio todavía tiene los `...` | Sustituid los `...` (4.2 o 4.6) y repetid |
| Los milisegundos del paso 3 no coinciden con los de otro compañero | Cada ordenador es distinto | Es normal: lo que importa es que el vectorizado sea **mucho** más rápido |

**En cualquier parte**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'tabla' is not defined` (o `np`…) | No habéis ejecutado la celda que crea esa variable, o Colab se reinició | Ejecutad las celdas **desde el paso 0** hasta la que falla, en orden |
| La comprobación da ❌ aunque creéis que está bien | Habéis corregido el código pero no habéis vuelto a ejecutar la celda del ejercicio | Ejecutad la celda del ejercicio **otra vez** y después la de comprobación |
| Sale un error al descargar el fichero (`URLError`, `FileNotFoundError`) | Sin internet o sin acceso a GitHub | Probad otra vez; si sigue, avisad al docente (puede dar el fichero `ocupacion_diaria.csv`) |
| Seguís sin verlo | A veces pasa | 1) Preguntad a vuestra pareja; 2) después, al docente; 3) **cuando hayáis terminado o agotado las pistas**, ejecutad la «Solución de referencia» del ejercicio. Primero intentadlo: lo que copiéis, tenéis que saber explicarlo |

## 🆘 Soluciones de referencia

- Ejecutad estas celdas **solo después de intentarlo**.
- Cada ejercicio tiene la suya.
- Si lo dejáis para después, ejecutadlas para poder seguir con el paso siguiente.

In [ ]:
#@title 🆘 Solución del ejercicio 1 · Arrays (ábrela solo después de intentarlo)
ocupacion = tabla[:, 0]
temperatura = tabla[:, 1]
festivo = tabla[:, 2]
precio = tabla[:, 3]
print("Solución del ejercicio 1 aplicada: ahora ejecutad la celda 2.4")

In [ ]:
#@title 🆘 Solución del ejercicio 3 · Vectorización (ábrela solo después de intentarlo)
ingreso_diario = ocupacion / 100 * 120 * precio
ocupacion_z = (ocupacion - ocupacion.mean()) / ocupacion.std()
print("Solución del ejercicio 3 aplicada: ahora ejecutad las celdas 4.3 y 4.7")

## Para reflexionar (con vuestra pareja)

**Parte A · Arrays**

1. ¿Qué diferencia hay entre escribir `tabla[:, 1]` y `tabla[1, :]`? *(probadlo)*
2. ¿Cuál es el `shape` de `tabla[:, 0]` y por qué?

**Parte B · Vectorización**

1. Si en vez de 1096 días hubiera 10 millones, ¿qué os parece que pasaría con el bucle del paso 3 y con la versión vectorizada?
2. ¿Qué cambiaríais en el cálculo del ingreso si el alojamiento tuviera 200 habitaciones? ¿Cuántos sitios del código hay que tocar?

## Qué habéis aprendido

**Parte A · Arrays**

- Un **array** tiene forma y tipo; se pide un trozo con `tabla[filas, columnas]`.
- Los índices empiezan en 0 y `:` significa «todo».
- Dar nombre a cada columna hace el código más claro.

**Parte B · Vectorización**

- **Vectorizar** = operar con todo el array a la vez, sin bucles: más corto y mucho más rápido.
- **Estandarizar** = restar la media y dividir entre la desviación típica: sirve para comparar cosas en la misma escala.